## Week 08 Excercise End to End Solution

## Import Required Libraries

This cell imports the main libraries needed for the Week 08 project.

In [ ]:
from __future__ import annotations

# Import standard libraries
import json
import os
import re

# Import dataclass utilities for structured state management
from dataclasses import dataclass, field

# Import typing helpers for clearer type annotations
from typing import Dict, Generator, List, Optional, Tuple

# Import Gradio for building the UI
import gradio as gr

# Import dotenv to load environment variables from a .env file
from dotenv import load_dotenv

# Import OpenAI client for model access
from openai import OpenAI

## Load Environment Variables and Verify API Keys

This cell loads environment variables and checks whether the required API credentials are available.

### 1. Load `.env` Configuration

The command:

loads environment variables from the `.env` file into the Python runtime.

The `override=True` option ensures that values in the `.env` file replace any existing environment variables with the same name.

This allows API keys and configuration settings to be stored securely outside the notebook code.

---

### 2. Retrieve API Keys

Two environment variables are read:

- **HUGGING_FACE_TOKEN**  
  Used for accessing Hugging Face models, datasets, or APIs.

- **OPENROUTER_API_KEY**  
  Used for accessing models through the OpenRouter API.

The values are retrieved using:

Only the first few characters are printed to confirm the key is loaded without exposing the full secret.

If the key is missing, the notebook prints a message indicating that it is not set.

---

### Summary

This step ensures that:

- environment variables are loaded correctly
- required API credentials are available
- sensitive keys remain protected while verifying configuration

It acts as a quick sanity check before making API calls later in the notebook.

In [ ]:
# Load environment variables from the .env file
load_dotenv(override=True)

# Retrieve tokens from environment variables
huggingface_token  = os.getenv("HUGGING_FACE_TOKEN")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")


# Check if Hugging Face token exists
if huggingface_token:
    # Print only the first few characters for security
    print(f"HuggingFace Token exists and begins with '{huggingface_token[:8]}'")
else:
    print("HuggingFace Token not set")


# Check if OpenRouter API key exists
if openrouter_api_key:
    # Print only the first few characters for security
    print(f"OpenRouter API Key exists and begins with '{openrouter_api_key[:7]}'")
else:
    print("OpenRouter API Key not set")

## Agent Configuration and LLM Response Handling

This cell defines helper utilities for configuring LLM agents and safely extracting text from model responses.

---

## 1. AgentConfig Dataclass

The `AgentConfig` class stores all configuration parameters needed to interact with a language model provider.

It includes:

- **name** – identifier for the agent
- **model** – model name used by the provider
- **api_key** – authentication key
- **base_url** – API endpoint
- **temperature** – randomness level of responses
- **supports_json** – whether the provider supports JSON responses
- **supports_temperature** – whether temperature control is supported

Using a dataclass simplifies configuration management and keeps agent settings organized.

---

## 2. Creating an OpenAI-Compatible Client

The function `make_client()` creates an OpenAI-compatible client object.

Many LLM providers support the OpenAI API format (including OpenRouter, Groq, and others).  
This function allows the same client interface to be used across different providers.

Steps performed:

1. Validate that an API key exists.
2. Create and return a configured `OpenAI` client instance.

This abstraction allows the system to switch providers easily without changing the rest of the code.

---

## 3. Extracting Text from LLM Responses

Different providers may return responses in slightly different formats.

The `extract_text()` function safely extracts the generated text regardless of format.

The function handles multiple cases:

### Case 1: Standard OpenAI Chat Response

### Case 2: Dictionary-style responses

Some SDKs return responses as dictionaries rather than objects.

### Case 3: Structured Content Blocks

Some providers return content as a list of structured parts:


In [ ]:
# Define a configuration object for each agent / LLM provider
@dataclass
class AgentConfig:
    """Holds all configuration required to talk to an LLM provider."""

    # Human-readable name for the agent
    name: str

    # Model identifier (e.g. gpt-4o-mini, llama3.2, etc.)
    model: str

    # API key used to authenticate with the provider
    api_key: str

    # Base API URL for the provider
    base_url: str

    # Default generation temperature
    temperature: float = 0.7

    # Whether the provider supports JSON structured output
    supports_json: bool = True

    # Whether the provider supports temperature configuration
    supports_temperature: bool = True


# Create an OpenAI-compatible client for any provider
def make_client(config: AgentConfig) -> OpenAI:
    """Return an OpenAI-compatible client for any provider."""

    # Ensure the API key exists before creating the client
    if not config.api_key:
        raise RuntimeError(f"Missing API key for agent '{config.name}'.")

    # Return a configured OpenAI-compatible client
    return OpenAI(api_key=config.api_key, base_url=config.base_url)


# Extract text content from an OpenAI-style LLM response
def extract_text(response) -> str:
    """Extract plain text from an OpenAI-style response object."""

    # Attempt to retrieve the "choices" field from the response
    choices = getattr(response, "choices", None) or (
        response.get("choices") if isinstance(response, dict) else None
    )

    if not choices:
        raise RuntimeError(f"LLM response missing choices: {response!r}")

    # Select the first response choice
    choice = choices[0]

    # Retrieve the message object
    message = getattr(choice, "message", None) or (
        choice.get("message") if isinstance(choice, dict) else None
    )

    content = None

    # Extract message content
    if message is not None:
        content = getattr(message, "content", None) or (
            message.get("content") if isinstance(message, dict) else None
        )

    # Handle structured responses (list of content blocks)
    if isinstance(content, list):
        parts: List[str] = []
        for part in content:
            if isinstance(part, dict):
                parts.append(
                    str(part.get("text") or part.get("output_text") or part.get("content", ""))
                )
            else:
                parts.append(str(part))
        content = "".join(parts)

    # Fallback for providers that return plain text directly
    if content is None:
        content = getattr(choice, "text", None) or (
            choice.get("text") if isinstance(choice, dict) else None
        )

    # Ensure content exists
    if content is None:
        raise RuntimeError(f"LLM response missing content: {response!r}")

    return str(content).strip()

## Multi-Agent Legal System Configuration

This cell defines the configuration for each agent participating in the **AI legal debate system**.

Each agent represents a different role in a simulated legal proceeding.

The system uses multiple LLM providers through **OpenAI-compatible APIs**.

---

# Provider Endpoints

Two provider gateways are used:

### HuggingFace Router
Used for open-source models.

This allows serverless access to models like:

- Llama-3.3-70B
- Mistral
- Qwen

---

### OpenRouter
Used to access proprietary models through one API.

Supported models include:

- Claude
- GPT models
- O-series reasoning models

---

# Agent Roles in the System

The system simulates a legal debate using specialized AI agents.

---

## 1. Plaintiff's Counsel

Represents the **claimant or user side of the case**.

Responsibilities:

- Argue why the plaintiff should win
- Present supporting reasoning
- Highlight damages or violations

Model used:

This allows more expressive and persuasive arguments.

---

## 2. Defense Counsel

Represents the **opposing side**.

Responsibilities:

- Challenge the plaintiff’s claims
- Provide counterarguments
- Highlight weaknesses in the case

Uses the same model as the plaintiff but operates independently.

---

## 3. Expert Witness

Provides **objective legal analysis**.

Responsibilities:

- Explain relevant legal principles
- Clarify facts
- Evaluate the strength of arguments

Model used:

Lower temperature ensures factual and precise responses.

---

## 4. Judge

The judge evaluates the arguments and makes a **final ruling**.

Responsibilities:

- Weigh both sides
- Consider expert testimony
- Provide a balanced judgment

Model used:

This reasoning model is optimized for deep evaluation.

Special notes:

- Does **not support JSON mode**
- Does **not use temperature**

---

## 5. Legal Strategist

Provides **meta-level strategy and legal insights**.

Responsibilities:

- Suggest improvements to arguments
- Identify missing evidence
- Provide structured reasoning

Model used:

Balanced between creativity and accuracy.

---

# System Architecture

The agents interact in a structured debate pipeline.


In [ ]:
# Base URLs for LLM providers

# HuggingFace Inference API endpoint that supports OpenAI-compatible requests
HF_BASE_URL = "https://router.huggingface.co/v1"

# OpenRouter endpoint which allows access to multiple models (Claude, GPT, etc.)
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"


# Plaintiff's Counsel Agent Configuration
# This agent represents the side arguing for the plaintiff (the user or claimant)
PLAINTIFF_CONFIG = AgentConfig(
    name="Plaintiff's Counsel",

    # Large instruction-tuned Llama model suitable for reasoning and argument generation
    model="meta-llama/Llama-3.3-70B-Instruct",

    # Authentication token for HuggingFace router
    api_key=huggingface_token,

    # Base API endpoint
    base_url=HF_BASE_URL,

    # Higher temperature allows more creative argument generation
    temperature=0.7,
)


# Defense Counsel Agent Configuration
# Represents the opposing side, generating counterarguments
DEFENSE_CONFIG = AgentConfig(
    name="Defense Counsel",

    # Same model family but treated as a separate independent agent
    model="meta-llama/Llama-3.3-70B-Instruct",

    api_key=huggingface_token,
    base_url=HF_BASE_URL,

    # Slight randomness encourages diverse counterarguments
    temperature=0.7,
)


# Expert Witness Agent Configuration
# This agent provides objective legal analysis instead of argumentation
EXPERT_CONFIG = AgentConfig(
    name="Expert Witness",

    # Claude Sonnet is used for strong analytical reasoning
    model="anthropic/claude-sonnet-4-5",

    api_key=openrouter_api_key,

    # Requests are routed through OpenRouter
    base_url=OPENROUTER_BASE_URL,

    # Lower temperature encourages precise and factual responses
    temperature=0.2,

    # Claude supports structured JSON outputs
    supports_json=True,
)


# Judge Agent Configuration
# This agent evaluates arguments and makes the final decision
JUDGE_CONFIG = AgentConfig(
    name="Judge",

    # OpenAI o3 reasoning model for careful analysis
    model="openai/o3",

    api_key=openrouter_api_key,
    base_url=OPENROUTER_BASE_URL,

    # Very low temperature for consistent reasoning
    temperature=0.1,

    # o3 reasoning models do not support JSON output mode
    supports_json=False,

    # o3 models ignore temperature parameters
    supports_temperature=False,
)


# Legal Strategist Agent Configuration
# Provides high-level legal strategy and structured reasoning
STRATEGIST_CONFIG = AgentConfig(
    name="Legal Strategist",

    # GPT-4o is used for structured planning and strategy suggestions
    model="openai/gpt-4o",

    api_key=openrouter_api_key,
    base_url=OPENROUTER_BASE_URL,

    # Moderate temperature balances creativity with accuracy
    temperature=0.4,

    # GPT-4o supports JSON structured responses
    supports_json=True,
)

## Multi-Agent Legal Roles and LLM Adapter

This block defines the core classes used in the Week 08 legal multi-agent system.

It includes:

- a reusable LLM adapter
- plaintiff and defense agents
- an expert witness
- a judge
- a legal strategist

Together, these classes create a structured legal reasoning workflow.

### 1. `LLMAdapter`

`LLMAdapter` is a thin wrapper around the OpenAI-compatible client.

Its job is to standardize model calls across providers.

It:
- builds chat messages
- applies system prompts
- handles temperature settings when supported
- optionally requests JSON output
- returns plain text using `extract_text()`

This makes the rest of the agent classes simple and provider-agnostic.

### 2. `PlaintiffCounsel`

This agent argues **for** the user's position.

Its system prompt defines it as an experienced plaintiff attorney whose goal is to:
- build the strongest claim
- cite legal theory
- identify favorable facts
- anticipate defense objections

The `argue()` method generates a persuasive plaintiff-side argument from:
- legal area
- case facts
- user position

### 3. `DefenseCounsel`

This agent argues **against** the user's position.

Its role is to stress-test the case by:
- identifying weaknesses
- raising procedural issues
- challenging evidence
- offering alternative fact interpretations

The `argue()` method creates the strongest counter-argument to the user's position.

### 4. `ExpertWitness`

This agent is neutral.

It does not advocate for either side.  
Instead, it provides objective legal analysis such as:
- key legal questions
- applicable law
- relevant precedents
- burden of proof
- critical risk factors

The `analyse()` method asks the model to return a JSON object.

Because LLMs sometimes return malformed JSON, the code includes fallback parsing logic:
- strip markdown fences
- search for embedded JSON with regex
- return a safe default structure if parsing fails

This makes the system more robust.

### 5. `Judge`

This agent evaluates both sides using a scoring rubric.

The rubric includes:
- legal theory strength
- factual support
- anticipation of counter-arguments
- procedural soundness
- overall persuasiveness

The `evaluate()` method:
- compares plaintiff and defense arguments
- includes expert analysis
- requests structured JSON scoring
- identifies vulnerabilities for both sides
- determines which side is stronger

It also contains fallback JSON parsing, similar to `ExpertWitness`.

### 6. `LegalStrategist`

This agent converts all prior outputs into practical user guidance.

It produces a structured case preparation memo with sections such as:
- strengths to leverage
- vulnerabilities to address
- evidence gaps
- recommended strategy
- immediate action items

Its role is not to judge the case, but to help the user prepare more effectively.

### Overall Flow

These classes support a pipeline like this:

1. Plaintiff builds the strongest favorable argument  
2. Defense responds with counterarguments  
3. Expert witness explains the legal landscape  
4. Judge evaluates both sides  
5. Legal strategist turns the full analysis into practical advice

This design creates a more balanced and useful legal reasoning system than relying on a single model response.

In [ ]:
# Adapter class that standardizes how all agents talk to an LLM
class LLMAdapter:
    """Thin wrapper around the OpenAI SDK; works with any OpenAI-compatible endpoint."""

    def __init__(self, config: AgentConfig):
        # Store the agent configuration
        self.config = config

        # Create an API client using the provider configuration
        self.client = make_client(config)

    def complete(
        self,
        prompt: str,
        *,
        system: Optional[str] = None,
        max_tokens: int = 600,
        json_mode: bool = False,
    ) -> str:
        # Build the chat messages list
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})

        # Base request parameters
        params: dict = dict(
            model=self.config.model,
            messages=messages,
            max_tokens=max_tokens,
        )

        # Only include temperature if the provider/model supports it
        if self.config.supports_temperature:
            params["temperature"] = self.config.temperature

        # Request JSON output only if both caller and provider support it
        if json_mode and self.config.supports_json:
            params["response_format"] = {"type": "json_object"}

        # Send request to the model and return plain text output
        response = self.client.chat.completions.create(**params)
        return extract_text(response)


# Plaintiff-side legal agent
class PlaintiffCounsel:
    """Constructs the strongest legal argument in favour of the user's position."""

    # Fixed system instruction that defines this agent's role and tone
    SYSTEM = (
        "You are a seasoned plaintiff's attorney. Your role is to construct "
        "the strongest possible legal argument in FAVOUR of your client's position. "
        "Be strategic, cite relevant legal principles, anticipate weaknesses, "
        "and write in a persuasive but professional legal tone."
    )

    def __init__(self, adapter: LLMAdapter):
        self.adapter = adapter

    def argue(self, case_description: str, legal_area: str, user_position: str) -> str:
        # Build a structured prompt containing legal area, facts, and user's position
        prompt = (
            f"Legal area: {legal_area}\n\n"
            f"Case facts:\n{case_description}\n\n"
            f"Your client's position: {user_position}\n\n"
            "Build the strongest possible argument for this position. Include:\n"
            "1. Core legal theory\n"
            "2. Key facts that support the claim\n"
            "3. Relevant legal principles or precedents\n"
            "4. Anticipated defence objections and pre-emptive rebuttals\n\n"
            "Be concise but thorough. Max 300 words."
        )

        # Generate plaintiff argument
        return self.adapter.complete(prompt, system=self.SYSTEM, max_tokens=500)


# Defense-side legal agent
class DefenseCounsel:
    """Mounts the strongest counter-argument to stress-test the user's position."""

    # Fixed system instruction defining the defense perspective
    SYSTEM = (
        "You are a sharp defense attorney. Your role is to dismantle the opposing "
        "party's legal position. Identify weaknesses, raise procedural issues, "
        "challenge the evidence, and present the strongest counter-arguments possible. "
        "Write in a professional legal tone."
    )

    def __init__(self, adapter: LLMAdapter):
        self.adapter = adapter

    def argue(self, case_description: str, legal_area: str, user_position: str) -> str:
        # Build prompt from the opposing party's position
        prompt = (
            f"Legal area: {legal_area}\n\n"
            f"Case facts:\n{case_description}\n\n"
            f"The opposing party's position: {user_position}\n\n"
            "Mount the strongest possible counter-argument. Include:\n"
            "1. Primary legal defence or counter-theory\n"
            "2. Factual weaknesses in the opposing case\n"
            "3. Procedural or evidentiary challenges\n"
            "4. Alternative interpretations of the facts\n\n"
            "Be incisive and precise. Max 300 words."
        )

        # Generate defense argument
        return self.adapter.complete(prompt, system=self.SYSTEM, max_tokens=500)


# Neutral expert legal analyst
class ExpertWitness:
    """Provides objective domain-specific analysis: statutes, precedents, burden of proof."""

    # System prompt enforcing objective, non-advocacy behavior
    SYSTEM = (
        "You are an independent legal expert and academic. You provide objective "
        "technical analysis of legal matters: relevant statutes, landmark case law, "
        "regulatory frameworks, and doctrinal issues. You do not advocate for either "
        "side — your role is to clarify the legal landscape and identify the key "
        "legal questions a court or regulator would focus on."
    )

    def __init__(self, adapter: LLMAdapter):
        self.adapter = adapter

    def analyse(
        self,
        case_description: str,
        legal_area: str,
        plaintiff_arg: str,
        defense_arg: str,
    ) -> Dict:
        # Ask the model to return structured legal analysis in JSON format
        prompt = (
            f"Legal area: {legal_area}\n\n"
            f"Case facts:\n{case_description}\n\n"
            f"Plaintiff's argument:\n{plaintiff_arg}\n\n"
            f"Defense's argument:\n{defense_arg}\n\n"
            "Provide an expert technical analysis as a JSON object:\n"
            "{\n"
            '  "key_legal_questions": ["question1", "question2"],\n'
            '  "applicable_law": "Summary of relevant statutes or doctrine",\n'
            '  "precedents": "Notable cases or rulings relevant to this matter",\n'
            '  "burden_of_proof": "Who bears the burden and what standard applies",\n'
            '  "critical_risk_factors": ["risk1", "risk2"]\n'
            "}\n"
            "Return only valid JSON."
        )

        raw = self.adapter.complete(prompt, system=self.SYSTEM, max_tokens=600, json_mode=True)

        # Try to parse the returned JSON safely
        try:
            clean = raw.strip()

            # Remove markdown fences if the model wrapped the JSON in code blocks
            if clean.startswith("```"):
                clean = clean.split("\n", 1)[-1]
            if clean.endswith("```"):
                clean = clean.rsplit("```", 1)[0]

            clean = clean.strip()
            return json.loads(clean)

        except Exception:
            # Fallback: try to extract the first JSON object from a messy response
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if m:
                try:
                    return json.loads(m.group())
                except Exception:
                    pass

            # Final fallback if parsing fails
            return {
                "key_legal_questions": [],
                "applicable_law": raw,
                "precedents": "",
                "burden_of_proof": "",
                "critical_risk_factors": [],
            }


# Judicial evaluation agent
class Judge:
    """Evaluates argument strength across legal criteria and surfaces vulnerabilities."""

    # Criteria used to score both sides
    RUBRIC = [
        "Legal theory strength",
        "Factual support",
        "Anticipation of counter-arguments",
        "Procedural soundness",
        "Overall persuasiveness",
    ]

    def __init__(self, adapter: LLMAdapter):
        self.adapter = adapter

    def evaluate(
        self,
        case_description: str,
        legal_area: str,
        plaintiff_arg: str,
        defense_arg: str,
        expert_analysis: Dict,
    ) -> Dict:
        # Convert rubric and expert analysis into prompt-ready text
        rubric_text = "\n".join(f"- {item}" for item in self.RUBRIC)
        expert_summary = json.dumps(expert_analysis, ensure_ascii=False, indent=2)

        # Ask the model to score both sides and return structured JSON
        prompt = (
            "You are an experienced judge evaluating the quality of legal arguments.\n\n"
            f"Legal area: {legal_area}\n\n"
            f"Case facts:\n{case_description}\n\n"
            f"Plaintiff's Counsel argues:\n{plaintiff_arg}\n\n"
            f"Defense Counsel argues:\n{defense_arg}\n\n"
            f"Expert legal analysis:\n{expert_summary}\n\n"
            "Score each side 0-10 on the following criteria:\n"
            f"{rubric_text}\n\n"
            "Return a JSON object:\n"
            "{\n"
            '  "stronger_position": "Plaintiff" or "Defense" or "Balanced",\n'
            '  "judicial_assessment": "Your overall assessment (2-3 sentences)",\n'
            '  "plaintiff_vulnerabilities": ["vuln1", "vuln2"],\n'
            '  "defense_vulnerabilities": ["vuln1", "vuln2"],\n'
            '  "scores": [\n'
            '    {"criterion": "...", "plaintiff": 0-10, "defense": 0-10, "notes": "..."}\n'
            "  ]\n"
            "}\n"
            "Return valid JSON only."
        )

        # o3 may need more output budget
        raw = self.adapter.complete(prompt, max_tokens=2000, json_mode=True)

        # Try to parse returned JSON
        try:
            clean = raw.strip()

            # Remove markdown fences if present
            if clean.startswith("```"):
                clean = clean.split("\n", 1)[-1]
            if clean.endswith("```"):
                clean = clean.rsplit("```", 1)[0]

            clean = clean.strip()
            data = json.loads(clean)

            if "scores" not in data:
                raise ValueError("scores missing")

            return data

        except Exception:
            # Fallback: extract JSON object from noisy text
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            if m:
                try:
                    data = json.loads(m.group())
                    if "scores" in data:
                        return data
                except Exception:
                    pass

            # Final fallback if parsing fails
            return {
                "stronger_position": "Unknown",
                "judicial_assessment": raw,
                "plaintiff_vulnerabilities": [],
                "defense_vulnerabilities": [],
                "scores": [],
            }


# Strategic summary agent
class LegalStrategist:
    """Synthesises all agent outputs into an actionable case preparation memo."""

    # System prompt focused on practical litigation guidance
    SYSTEM = (
        "You are a senior legal strategist with 30 years of litigation experience. "
        "You synthesise complex legal analysis into clear, actionable advice. "
        "Your goal is to help the user strengthen their case and prepare for "
        "the arguments they will face. Be practical, specific, and candid."
    )

    def __init__(self, adapter: LLMAdapter):
        self.adapter = adapter

    def advise(
        self,
        case_description: str,
        legal_area: str,
        user_position: str,
        plaintiff_arg: str,
        defense_arg: str,
        expert_analysis: Dict,
        judge_result: Dict,
    ) -> str:
        # Build a strategy memo request using all prior agent outputs
        prompt = (
            f"Legal area: {legal_area}\n"
            f"User's position: {user_position}\n\n"
            f"Case facts:\n{case_description}\n\n"
            f"Plaintiff's argument:\n{plaintiff_arg}\n\n"
            f"Defense's argument:\n{defense_arg}\n\n"
            f"Expert analysis:\n{json.dumps(expert_analysis, indent=2)}\n\n"
            f"Judicial assessment:\n{json.dumps(judge_result, indent=2)}\n\n"
            "Provide a structured case preparation memo:\n\n"
            "STRENGTHS TO LEVERAGE\n"
            "- List 3 strongest points in the user's favour\n\n"
            "VULNERABILITIES TO ADDRESS\n"
            "- List 3 key weaknesses the user must shore up\n\n"
            "EVIDENCE GAPS\n"
            "- What additional evidence or documentation should be gathered?\n\n"
            "RECOMMENDED STRATEGY\n"
            "- Concise recommended litigation or settlement strategy\n\n"
            "IMMEDIATE ACTION ITEMS\n"
            "- 3 to 5 concrete next steps for case preparation\n\n"
            "Max 400 words. Be direct and practical."
        )

        # Generate the final strategic advice memo
        return self.adapter.complete(prompt, system=self.SYSTEM, max_tokens=600)

## Agent Initialization, Output Formatting, and Case Analysis Pipeline

This block wires together the legal agents, formats their outputs for display, and defines the main end-to-end case analysis workflow.

### 1. Agent Initialization

The first lines create one instance of each legal role:

- **Plaintiff Counsel**
- **Defense Counsel**
- **Expert Witness**
- **Judge**
- **Legal Strategist**

Each role is connected to its own `LLMAdapter`, which uses the provider/model configuration defined earlier.

This means every agent can behave independently, even if some share the same underlying model family.

### 2. Formatting Helper Functions

Two helper functions turn structured JSON-style outputs into readable markdown.

#### `_format_expert()`
This function formats the expert witness analysis into sections such as:

- Key Legal Questions
- Applicable Law
- Relevant Precedents
- Burden of Proof
- Critical Risk Factors

This makes the expert output easy to display in the UI.

#### `_format_judge()`
This function formats the judge’s structured decision into readable sections:

- stronger position
- judicial assessment
- plaintiff vulnerabilities
- defense vulnerabilities

It converts the judge’s JSON response into presentation-ready markdown.

### 3. Legal Area Options

`LEGAL_AREAS` contains a predefined list of common legal domains such as:

- Contract Law
- Employment Law
- Intellectual Property
- Family Law
- Data Privacy / GDPR

This is likely used in the UI dropdown so users can classify the type of legal issue they are describing.

### 4. `run_case_analysis()` Pipeline

This is the main orchestration function.

It is written as a **generator**, which means it can stream intermediate results step by step instead of waiting until everything is complete.

That improves the user experience in a Gradio interface because the user can see progress as each agent finishes.

### 5. Input Validation

Before running any agent, the function checks whether the user has entered:

- a case description
- a position in the case

If either is missing, it yields a helpful message and stops.

### 6. Step-by-Step Multi-Agent Workflow

The function then runs the legal analysis in five stages:

#### Step 1 — Plaintiff Argument
The plaintiff agent builds the strongest possible argument in favor of the user’s position.

#### Step 2 — Defense Argument
The defense agent stress-tests the case by generating counterarguments.

#### Step 3 — Expert Analysis
The expert witness provides neutral legal analysis and the result is formatted into markdown.

#### Step 4 — Judicial Evaluation
The judge compares both sides, scores them using the rubric, and returns:
- stronger side
- assessment
- vulnerabilities
- criterion-level scores

The score entries are also converted into table rows for UI display.

#### Step 5 — Strategic Advice
The legal strategist produces a practical case-preparation memo based on everything generated so far.

### 7. Final Summary

At the end, the function builds a short summary showing:

- the overall assessment
- which side appears stronger

This gives the user a quick high-level takeaway in addition to the full detailed outputs.

### 8. Why a Generator is Used

Because `run_case_analysis()` uses `yield`, the UI can update progressively while the pipeline runs.

That means the user can see statuses like:

- Plaintiff's Counsel is building your argument
- Defense Counsel is preparing counter-arguments
- Expert Witness is analysing applicable law

This makes the application feel more interactive and transparent.

### Overall Purpose

This block is the core orchestration layer of the Week 08 legal assistant.

It connects:
- the role-based agents
- the formatting helpers
- the streamed execution pipeline

Together, these pieces create a multi-stage legal reasoning system that produces arguments, counterarguments, legal analysis, judicial scoring, and practical strategy advice.

In [ ]:
# Create one instance of each legal agent using its configured LLM adapter
plaintiff_counsel = PlaintiffCounsel(LLMAdapter(PLAINTIFF_CONFIG))
defense_counsel   = DefenseCounsel(LLMAdapter(DEFENSE_CONFIG))
expert_witness    = ExpertWitness(LLMAdapter(EXPERT_CONFIG))
judge             = Judge(LLMAdapter(JUDGE_CONFIG))
strategist        = LegalStrategist(LLMAdapter(STRATEGIST_CONFIG))


# Format the expert witness JSON output into readable markdown
def _format_expert(expert: Dict) -> str:
    lines = []

    # Add key legal questions
    if ql := expert.get("key_legal_questions"):
        lines.append("**Key Legal Questions**")
        lines.extend(f"- {q}" for q in ql)
        lines.append("")

    # Add applicable law summary
    if al := expert.get("applicable_law"):
        lines.append(f"**Applicable Law**\n{al}\n")

    # Add precedent discussion
    if pr := expert.get("precedents"):
        lines.append(f"**Relevant Precedents**\n{pr}\n")

    # Add burden of proof explanation
    if bp := expert.get("burden_of_proof"):
        lines.append(f"**Burden of Proof**\n{bp}\n")

    # Add critical risk factors
    if rf := expert.get("critical_risk_factors"):
        lines.append("**Critical Risk Factors**")
        lines.extend(f"- {r}" for r in rf)

    return "\n".join(lines)


# Format the judge's structured output into readable markdown
def _format_judge(result: Dict) -> str:
    lines = []

    # Show which side appears stronger
    if sp := result.get("stronger_position"):
        lines.append(f"**Stronger Position:** {sp}\n")

    # Add the judge's overall narrative assessment
    if ja := result.get("judicial_assessment"):
        lines.append(f"**Judicial Assessment**\n{ja}\n")

    # Add vulnerabilities on the plaintiff side
    if pv := result.get("plaintiff_vulnerabilities"):
        lines.append("**Plaintiff Vulnerabilities**")
        lines.extend(f"- {v}" for v in pv)
        lines.append("")

    # Add vulnerabilities on the defense side
    if dv := result.get("defense_vulnerabilities"):
        lines.append("**Defense Vulnerabilities**")
        lines.extend(f"- {v}" for v in dv)

    return "\n".join(lines)


# Predefined legal area choices for the UI
LEGAL_AREAS = [
    "Contract Law",
    "Employment Law",
    "Intellectual Property",
    "Corporate / M&A",
    "Regulatory / Compliance",
    "Personal Injury / Tort",
    "Real Estate / Property",
    "Data Privacy / GDPR",
    "Criminal Law",
    "Family Law",
    "Other",
]


# Main pipeline function that runs the multi-agent legal analysis step by step
def run_case_analysis(
    case_description: str,
    legal_area: str,
    user_position: str,
) -> Generator:
    """Streams results step by step as each agent completes its work."""

    # Default empty UI output placeholders
    empty = ("", "", "", "", [], "", "")

    # Validate required inputs
    if not case_description.strip():
        yield ("Please describe your case to begin.", *empty[1:])
        return

    if not user_position.strip():
        yield ("Please describe your position in this case.", *empty[1:])
        return

    # Step 1 — Plaintiff builds the strongest argument for the user's position
    yield ("Plaintiff's Counsel is building your argument...", "", "", "", [], "", "")
    p_arg = plaintiff_counsel.argue(case_description, legal_area, user_position)

    # Step 2 — Defense generates the strongest counter-argument
    yield (p_arg, "Defense Counsel is preparing counter-arguments...", "", "", [], "", "")
    d_arg = defense_counsel.argue(case_description, legal_area, user_position)

    # Step 3 — Expert witness provides objective legal analysis
    yield (p_arg, d_arg, "Expert Witness is analysing applicable law and precedents...", "", [], "", "")
    expert = expert_witness.analyse(case_description, legal_area, p_arg, d_arg)
    expert_md = _format_expert(expert)

    # Step 4 — Judge evaluates both sides using the rubric
    yield (p_arg, d_arg, expert_md, "Judge is evaluating argument strength...", [], "", "")
    judge_result = judge.evaluate(case_description, legal_area, p_arg, d_arg, expert)

    # Convert judge score objects into table rows for UI display
    score_rows = [
        [e.get("criterion", ""), e.get("plaintiff", ""), e.get("defense", ""), e.get("notes", "")]
        for e in judge_result.get("scores", [])
    ]
    judge_md = _format_judge(judge_result)

    # Step 5 — Legal strategist prepares practical next-step advice
    yield (p_arg, d_arg, expert_md, judge_md, score_rows, "Legal Strategist is preparing your case memo...", "")
    strategy = strategist.advise(
        case_description, legal_area, user_position,
        p_arg, d_arg, expert, judge_result,
    )

    # Build a short top-level summary for the final UI section
    stronger = judge_result.get("stronger_position", "Unknown")
    summary_md = (
        f"**Overall Assessment:** {judge_md.split(chr(10))[0]}\n\n"
        f"**Stronger Position:** {stronger}"
    )

    # Final streamed result containing all completed outputs
    yield (p_arg, d_arg, expert_md, judge_md, score_rows, strategy, summary_md)

## Gradio UI for the Legal Counsel Agents App

This block builds and launches the complete Gradio interface for the Week 08 legal multi-agent system.

### 1. Theme and Styling

The UI uses a custom Gradio theme:

- primary hue: slate
- secondary hue: blue
- neutral hue: gray

A large custom CSS block is also defined to create a courtroom-inspired visual style.

The CSS controls:

- dark background colors
- serif typography for a formal legal look
- custom panel colors for each legal role
- styled primary action button
- hidden footer

Each agent output panel has its own visual identity:

- **Plaintiff panel** → blue
- **Defense panel** → red
- **Expert panel** → amber
- **Judge panel** → gray
- **Strategy panel** → green

This makes the app easier to scan and gives each role a clear visual distinction.

### 2. Main Application Container

The app is created using `gr.Blocks(...)`.

Important options include:

- `title="Counsel Agents"` for the browser tab
- `fill_width=True` to let the layout use more horizontal space
- `theme=THEME` for the default theme colors
- `css=CUSTOM_CSS` for the custom visual design

### 3. Header and Introduction

A markdown heading introduces the app as:

**Legal Counsel Agents**

It explains that five AI legal specialists will analyze the case from different perspectives.

This gives the user immediate context for what the tool does.

### 4. Input Section

The top input area collects three pieces of information:

#### Case Description
A large textbox where the user enters the facts of the case.

This is intended for:
- dates
- parties
- actions taken
- documents
- communications

#### Legal Area
A dropdown containing predefined legal categories such as:
- Contract Law
- Employment Law
- Criminal Law
- Family Law

#### Your Position
A smaller textbox where the user states their side of the case, such as:
- plaintiff seeking damages
- defendant contesting liability

These three inputs are passed into the multi-agent analysis pipeline.

### 5. Action Button

The button:

**Analyse My Case**

starts the full multi-agent workflow when clicked.

It is styled as the primary action button and uses the large button size to make it prominent.

### 6. Output Layout

The results are displayed in several sections.

#### Summary Area
A top markdown block shows a high-level final summary after analysis.

#### Row 1: Competing Arguments
Two side-by-side panels show:

- Plaintiff's Counsel
- Defense Counsel

This lets the user compare both sides directly.

#### Row 2: Legal Analysis and Judgment
Two additional panels show:

- Expert Witness Analysis
- Judge's Assessment

This separates neutral legal analysis from judicial evaluation.

#### Row 3: Strategy and Scores
The final row contains:

- a **Strategic Case Memo** generated by the legal strategist
- an **Argument Scores** table showing judge scoring across criteria

The table includes columns for:
- criterion
- plaintiff score
- defense score
- notes

### 7. Legal Disclaimer

A disclaimer is shown at the bottom stating that the tool is:

- for legal research and preparation only
- not legal advice
- not a substitute for a qualified attorney

This is especially important in legal applications.

### 8. Button Click Wiring

The `run_button.click(...)` call connects the UI to the backend function `run_case_analysis`.

Inputs:
- case description
- legal area
- user position

Outputs:
- plaintiff output
- defense output
- expert output
- judge output
- score table
- strategy memo
- summary

Because `run_case_analysis()` is a generator, the UI can update progressively as each agent finishes.

### 9. Queueing and Launch

Finally, the app enables request queueing using:

- `default_concurrency_limit=4`

This allows multiple requests to be handled safely without overwhelming the backend.

Then the app is launched with:

`demo.queue(...).launch()`

This starts the Gradio server and opens the interface for use.

### Overall Purpose

This block turns the entire Week 08 legal multi-agent system into a polished interactive application.

It combines:

- structured user inputs
- role-based agent outputs
- streaming updates
- scoring tables
- strategic summaries
- professional styling

This is the final UI layer that makes the system usable end to end.

In [ ]:
# Define the overall Gradio theme colors used across the app
THEME = gr.themes.Default(
    primary_hue="slate",
    secondary_hue="blue",
    neutral_hue="gray",
)

# Custom CSS used to create a dark, courtroom-style visual design
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=Source+Serif+4:ital,wght@0,300;0,400;1,300&display=swap');

body, .gradio-container {
    background: #0e0e0e;
    color: #e8e2d9;
    font-family: 'Source Serif 4', Georgia, serif;
}
h1, h2, h3 { font-family: 'Playfair Display', Georgia, serif !important; }
.gradio-container { max-width: 1400px !important; }

#plaintiff-panel {
    background: linear-gradient(160deg, #0a1628 0%, #0d1f3c 100%);
    border-left: 4px solid #3b82f6;
    border-radius: 4px 12px 12px 4px;
    padding: 20px; color: #bfdbfe;
    box-shadow: inset 0 0 60px rgba(59,130,246,0.05), 0 8px 32px rgba(0,0,0,0.4);
}
#defense-panel {
    background: linear-gradient(160deg, #1a0a0a 0%, #2d0f0f 100%);
    border-left: 4px solid #ef4444;
    border-radius: 4px 12px 12px 4px;
    padding: 20px; color: #fecaca;
    box-shadow: inset 0 0 60px rgba(239,68,68,0.05), 0 8px 32px rgba(0,0,0,0.4);
}
#expert-panel {
    background: linear-gradient(160deg, #1a1400 0%, #2d2200 100%);
    border-left: 4px solid #f59e0b;
    border-radius: 4px 12px 12px 4px;
    padding: 20px; color: #fde68a;
    box-shadow: inset 0 0 60px rgba(245,158,11,0.05), 0 8px 32px rgba(0,0,0,0.4);
}
#judge-panel {
    background: linear-gradient(160deg, #111111 0%, #1a1a1a 100%);
    border-left: 4px solid #9ca3af;
    border-radius: 4px 12px 12px 4px;
    padding: 20px; color: #e5e7eb;
    box-shadow: inset 0 0 60px rgba(156,163,175,0.05), 0 8px 32px rgba(0,0,0,0.4);
}
#strategy-panel {
    background: linear-gradient(160deg, #001a0d 0%, #002d15 100%);
    border-left: 4px solid #10b981;
    border-radius: 4px 12px 12px 4px;
    padding: 20px; color: #a7f3d0;
    box-shadow: inset 0 0 60px rgba(16,185,129,0.05), 0 8px 32px rgba(0,0,0,0.4);
}
.gr-button-primary {
    background: linear-gradient(135deg, #1e3a5f, #1d4ed8) !important;
    border: 1px solid #3b82f6 !important;
    font-family: 'Playfair Display', serif !important;
    letter-spacing: 0.05em !important;
    font-size: 1rem !important;
    padding: 12px 32px !important;
}
.gr-button-primary:hover {
    background: linear-gradient(135deg, #1d4ed8, #2563eb) !important;
    box-shadow: 0 0 20px rgba(59,130,246,0.3) !important;
}
footer { display: none !important; }
"""

# Build the main Gradio application
with gr.Blocks(
    title="Counsel Agents",
    fill_width=True,
    theme=THEME,
    css=CUSTOM_CSS,
) as demo:

    # Application title and short description
    gr.Markdown(
        "# Legal Counsel Agents\n"
        "*Five AI legal specialists analyse your case from every angle "
        "so you walk into proceedings fully prepared.*"
    )

    # Input section: case description, legal area, and user's legal position
    with gr.Row():
        with gr.Column(scale=2):
            case_input = gr.Textbox(
                label="Case Description",
                placeholder=(
                    "Describe the facts of your case in as much detail as possible. "
                    "Include key dates, parties involved, actions taken, and any "
                    "relevant documents or communications..."
                ),
                lines=7,
            )
        with gr.Column(scale=1):
            legal_area_input = gr.Dropdown(
                label="Legal Area",
                choices=LEGAL_AREAS,
                value="Contract Law",
            )
            position_input = gr.Textbox(
                label="Your Position",
                placeholder=(
                    "e.g. 'I am the plaintiff seeking damages for breach of contract' "
                    "or 'We are the defendant arguing the contract was void ab initio'"
                ),
                lines=4,
            )

    # Main action button that starts the multi-agent analysis
    run_button = gr.Button("Analyse My Case", variant="primary", size="lg")

    # Summary area shown near the top after analysis
    gr.Markdown("---")
    summary_md = gr.Markdown("")
    gr.Markdown("---")

    # First output row: plaintiff and defense arguments
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Plaintiff's Counsel")
            plaintiff_out = gr.Markdown(
                "Argument will appear here...", elem_id="plaintiff-panel"
            )
        with gr.Column():
            gr.Markdown("### Defense Counsel")
            defense_out = gr.Markdown(
                "Counter-argument will appear here...", elem_id="defense-panel"
            )

    gr.Markdown("---")

    # Second output row: expert analysis and judge assessment
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Expert Witness Analysis")
            expert_out = gr.Markdown(
                "Legal analysis will appear here...", elem_id="expert-panel"
            )
        with gr.Column():
            gr.Markdown("### Judge's Assessment")
            judge_out = gr.Markdown(
                "Judicial evaluation will appear here...", elem_id="judge-panel"
            )

    gr.Markdown("---")

    # Third output row: legal strategy memo and scoring table
    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown("### Strategic Case Memo")
            strategy_out = gr.Markdown(
                "Your preparation memo will appear here...", elem_id="strategy-panel"
            )
        with gr.Column(scale=1):
            gr.Markdown("### Argument Scores")
            score_table = gr.Dataframe(
                headers=["Criterion", "Plaintiff", "Defense", "Notes"],
                datatype=["str", "number", "number", "str"],
                interactive=False,
            )

    # Legal disclaimer at the bottom of the interface
    gr.Markdown(
        "*This tool is for legal research and preparation purposes only. "
        "It does not constitute legal advice. Always consult a qualified attorney.*"
    )

    # Connect the button to the streamed multi-agent pipeline
    run_button.click(
        fn=run_case_analysis,
        inputs=[case_input, legal_area_input, position_input],
        outputs=[
            plaintiff_out, defense_out, expert_out,
            judge_out, score_table, strategy_out, summary_md,
        ],
        queue=True,
    )

# Enable request queueing and launch the Gradio app
demo.queue(default_concurrency_limit=4).launch()